# Fabric Notebook URL Scanner

Scans every Fabric notebook in every workspace this identity can see, extracts
URLs from the notebook source (and any embedded markdown), and writes findings
to a Delta table in the attached Lakehouse.

## Architecture

1. **Driver — enumerate workspaces + notebooks** in parallel (asyncio).
   - `ADMIN_MODE=True` → Fabric admin scanner (up to 100 workspaces per call,
     16 concurrent scans, ~3–5 min for 10k workspaces).
   - `ADMIN_MODE=False` → per-workspace item listing (slower; only works for
     workspaces the caller can access).
2. **Spark — fan out `getDefinition` calls** with `mapPartitions`.
   - Each executor partition runs its own asyncio loop with a bounded
     semaphore (`EXECUTOR_CONCURRENCY`) and full LRO polling for the 202
     responses returned by `getDefinition`.
3. **Scan + persist** — every notebook part is base64-decoded, parsed
   (`.ipynb` cells + outputs, `.py` source, `.md`), and matched against a
   URL regex. Findings land in `OUTPUT_TABLE`.

## Required permissions

- Token audience: `pbi` (works for both `api.powerbi.com` and
  `api.fabric.microsoft.com`).
- `Item.Read.All` on workspaces in scope **or** Fabric Administrator role
  if `ADMIN_MODE=True` (and the *"Service principals can call read-only
  admin APIs"* tenant setting if running as SP).

## Tuning

- For 10k workspaces / 18k notebooks on a Small pool, start with
  `EXECUTOR_CONCURRENCY=60`, `TARGET_PARTITION_SIZE=200`.
- Bump executor count up to the point where you start seeing `429` errors
  in the `error` column of the output table, then back off.

In [ ]:
# === Configuration ===========================================================

# -----------------------------------------------------------------------------
# READ FROM — which workspaces & notebooks to scan
# -----------------------------------------------------------------------------
# Source mode: where the notebook *content* comes from.
#   "api"       => enumerate via Fabric REST APIs and call getDefinition for
#                  each notebook. Needs admin or Item.Read.All. Slower (LROs).
#   "lakehouse" => read pre-exported notebook files (.ipynb / .py / .md)
#                  directly from a Lakehouse Files path. No REST calls, no
#                  LROs — usually 10–100x faster when an export already exists
#                  (Git integration, fabric-cli, or your own pipeline).
SOURCE_MODE = "api"

# --- API-mode config (used only when SOURCE_MODE == "api") -------------------
# ADMIN_MODE=True uses the Fabric admin scanner API to enumerate the entire
# tenant. Requires Fabric Administrator + (for SP) the tenant setting
# "Service principals can call read-only admin APIs".
# ADMIN_MODE=False enumerates only workspaces the caller can list.
ADMIN_MODE = True

# Optional: restrict the scan to specific workspace GUIDs.
#   None             -> all workspaces returned by the chosen enumeration mode
#   ["guid1", ...]   -> only these workspace IDs
READ_WORKSPACE_IDS = None

# Test cap on notebook count. None = full run; integer = smoke test cap.
MAX_NOTEBOOKS = None

# --- Lakehouse-mode config (used only when SOURCE_MODE == "lakehouse") -------
# Leave the two IDs blank to read from the Lakehouse currently attached to
# this notebook (Fabric mounts it at the working directory, so the SUBPATH
# resolves against it). Set them explicitly to scan a Lakehouse in a different
# workspace.
SOURCE_LAKEHOUSE_WORKSPACE_ID   = ""        # blank => attached workspace
SOURCE_LAKEHOUSE_WORKSPACE_NAME = ""        # purely informational; copied
                                            # into the workspace_name column
SOURCE_LAKEHOUSE_ID             = ""        # blank => attached default lakehouse
SOURCE_SUBPATH                  = "Files/notebooks"   # under the lakehouse root
# Spark binaryFile glob. Single-pattern only. To pick up .py / .md files too,
# try "*.{ipynb,py,md}" — works on most Hadoop versions Fabric ships.
SOURCE_FILE_GLOB              = "*.ipynb"

# -----------------------------------------------------------------------------
# WRITE TO — where the inventory + findings land
# -----------------------------------------------------------------------------
# Set WRITE_TO_DEFAULT_LAKEHOUSE=True to write to the Lakehouse currently
# attached to this notebook (simplest). Set False to write to an explicit
# Lakehouse in any workspace via ABFSS — useful for cross-workspace audits.
WRITE_TO_DEFAULT_LAKEHOUSE = True

# Used only when WRITE_TO_DEFAULT_LAKEHOUSE = False:
WRITE_WORKSPACE_ID  = "<output-workspace-guid>"
WRITE_LAKEHOUSE_ID  = "<output-lakehouse-guid>"
WRITE_LAKEHOUSE_NAME = "<output-lakehouse-name>"   # used only for SQL hints

# Schema (optional; leave None for the lakehouse default "dbo" / unschemed).
WRITE_SCHEMA = None

# Table names.
INVENTORY_TABLE = "notebook_inventory"
OUTPUT_TABLE    = "notebook_url_scan"

# -----------------------------------------------------------------------------
# Tuning
# -----------------------------------------------------------------------------
TARGET_PARTITION_SIZE = 200          # notebooks per Spark partition
EXECUTOR_CONCURRENCY  = 60           # concurrent in-flight HTTP per partition
MAX_RETRIES           = 6            # per-request retry budget on 429/5xx
TOKEN_AUDIENCE        = "pbi"        # token audience alias (Fabric/Power BI)

# REST endpoints.
FABRIC_BASE = "https://api.fabric.microsoft.com"
PBI_BASE    = "https://api.powerbi.com"


In [ ]:
# === Resolve read/write targets ==============================================
# Builds the fully-qualified ABFSS paths (when writing to an explicit Lakehouse)
# and prints the effective read/write configuration so it is obvious at run
# time exactly which tenant/workspace/lakehouse this notebook will touch.

ONELAKE_HOST = "onelake.dfs.fabric.microsoft.com"


def _table_path(workspace_id: str, lakehouse_id: str, table: str,
                schema: str | None = None) -> str:
    if schema:
        return (f"abfss://{workspace_id}@{ONELAKE_HOST}/"
                f"{lakehouse_id}/Tables/{schema}/{table}")
    return (f"abfss://{workspace_id}@{ONELAKE_HOST}/"
            f"{lakehouse_id}/Tables/{table}")


def _files_path(workspace_id: str, lakehouse_id: str, subpath: str) -> str:
    sub = (subpath or "").lstrip("/")
    return (f"abfss://{workspace_id}@{ONELAKE_HOST}/"
            f"{lakehouse_id}/{sub}" if sub
            else f"abfss://{workspace_id}@{ONELAKE_HOST}/{lakehouse_id}")


if WRITE_TO_DEFAULT_LAKEHOUSE:
    inventory_target = INVENTORY_TABLE
    output_target    = OUTPUT_TABLE
    write_kind = "saveAsTable (default attached Lakehouse)"
else:
    inventory_target = _table_path(WRITE_WORKSPACE_ID, WRITE_LAKEHOUSE_ID,
                                   INVENTORY_TABLE, WRITE_SCHEMA)
    output_target    = _table_path(WRITE_WORKSPACE_ID, WRITE_LAKEHOUSE_ID,
                                   OUTPUT_TABLE, WRITE_SCHEMA)
    write_kind = f"ABFSS Delta path (workspace {WRITE_WORKSPACE_ID})"


# Resolve the source URI for lakehouse mode. Blank IDs => relative path
# against the attached default lakehouse (Fabric mounts it at the cwd).
if SOURCE_MODE == "lakehouse":
    if SOURCE_LAKEHOUSE_WORKSPACE_ID and SOURCE_LAKEHOUSE_ID:
        source_uri = _files_path(SOURCE_LAKEHOUSE_WORKSPACE_ID,
                                 SOURCE_LAKEHOUSE_ID, SOURCE_SUBPATH)
        source_kind_label = (f"ABFSS Lakehouse path "
                             f"(workspace {SOURCE_LAKEHOUSE_WORKSPACE_ID})")
    else:
        source_uri = (SOURCE_SUBPATH or "Files")
        source_kind_label = "attached default Lakehouse (relative)"
else:
    source_uri = None
    source_kind_label = (("admin scanner (all workspaces)" if ADMIN_MODE
                          else "user-accessible workspaces"))


print("============================================================")
print(" Fabric Notebook URL Scanner — runtime configuration")
print("============================================================")
print(f" SOURCE MODE : {SOURCE_MODE}")
print(f" READ FROM   : {source_kind_label}")
if SOURCE_MODE == "lakehouse":
    print(f"   path      : {source_uri}")
    print(f"   glob      : {SOURCE_FILE_GLOB}")
elif READ_WORKSPACE_IDS:
    print(f"               restricted to {len(READ_WORKSPACE_IDS)} workspace(s)")
if MAX_NOTEBOOKS:
    print(f"               capped at {MAX_NOTEBOOKS} notebooks (test mode)")
print(f" WRITE TO    : {write_kind}")
print(f"   inventory : {inventory_target}")
print(f"   findings  : {output_target}")
if SOURCE_MODE == "api":
    print(f" CONCURRENCY : {EXECUTOR_CONCURRENCY}/executor "
          f"x partition size {TARGET_PARTITION_SIZE}")
else:
    print(f" PARTITIONS  : ~{TARGET_PARTITION_SIZE} files per Spark partition")
print("============================================================")


In [ ]:
# === Imports + token =========================================================
import asyncio
import base64
import json
import re
import time
from datetime import datetime, timezone
from typing import Iterable, Iterator, List, Tuple

import aiohttp
import requests
from pyspark.sql import Row
from pyspark.sql.types import (
    BooleanType, StringType, StructField, StructType, TimestampType
)

try:
    import notebookutils
    _get_token = lambda aud: notebookutils.credentials.getToken(aud)
except ImportError:
    import mssparkutils
    _get_token = lambda aud: mssparkutils.credentials.getToken(aud)


def get_token() -> str:
    return _get_token(TOKEN_AUDIENCE)


_token = get_token()
print(f"Acquired Fabric token ({len(_token)} chars).")


In [ ]:
# === URL extraction + direction classification + secret scanning =============
# Driver-side reference. The executor inlines its own copies inside
# `fetch_partition` (Spark closures + 3rd-party imports across workers).

# Cloud / Hadoop URI schemes are matched explicitly (abfss, wasbs, s3a, gs,
# dbfs, adl, hdfs) so paths like
#   abfss://container@account.dfs.core.windows.net/path
# are recognized as URLs by the scanner, not just http(s).
URL_RE = re.compile(
    r"(?i)\b(?:https?|ftps?|file|abfss?|wasbs?|s3a?|s3n|gs|dbfs|adl|hdfs)://"
    r"[^\s'\"<>`)\]}]+"
    r"|\bwww\.[^\s'\"<>`)\]}]+\.[^\s'\"<>`)\]}]+"
)
TRAILING = ".,;:!?)]}>\"'`"

# A read-style I/O call: the URL is being fetched FROM.
READ_RE = re.compile(
    r"(?:"
    r"\brequests?\.(?:get|head|options)\s*\("
    r"|\burllib(?:\.request)?\.urlopen\s*\("
    r"|\burlretrieve\s*\("
    r"|\b(?:pd|pandas|pl|polars)\.(?:read|scan)_\w+\s*\("
    r"|\.read_\w+\s*\("
    r"|\.scan_(?:parquet|csv|ndjson|delta|ipc|arrow)\s*\("
    r"|\bspark\.read\b"
    r"|\b(?:httpx|aiohttp)\.(?:get|head|options)\s*\("
    r"|\bsession\.(?:get|head|options)\s*\("
    r"|\bfetch\s*\("
    r"|\bsc\.textFile\s*\("
    r"|\bspark\.sparkContext\.textFile\s*\("
    r"|!\s*wget\b"
    r"|!\s*curl\b(?![^\n]*-X\s+(?:POST|PUT|PATCH|DELETE))"
    r"|\bdownload\w*\s*\("
    r")",
    re.IGNORECASE,
)

# A write-style I/O call: the URL is being written/posted TO.
# Adds polars `.write_<fmt>(...)` (e.g. df.write_delta, df.write_parquet)
# alongside the existing pandas `.to_<fmt>(...)` patterns.
WRITE_RE = re.compile(
    r"(?:"
    r"\brequests?\.(?:post|put|patch|delete)\s*\("
    r"|\b(?:httpx|aiohttp)\.(?:post|put|patch|delete)\s*\("
    r"|\bsession\.(?:post|put|patch|delete)\s*\("
    r"|\b(?:pd|pandas)\.to_\w+\s*\("
    r"|\.to_(?:csv|json|parquet|sql|excel|html|hdf|pickle|orc|feather)\s*\("
    r"|\.write_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|database)\s*\("
    r"|\.write\.\w+"
    r"|\.save(?:AsTable|AsTextFile)?\s*\("
    r"|\bupload\w*\s*\("
    r"|\bput_object\s*\("
    r"|!\s*curl[^\n]*-X\s+(?:POST|PUT|PATCH|DELETE)"
    r"|!\s*curl[^\n]*(?:--upload-file|--data\b|-d\s)"
    r")",
    re.IGNORECASE,
)

# --- Notebook-level "potential writer" detection -----------------------------
# These match WRITE-style CALLS regardless of whether the destination is a
# literal URL. They let us flag a notebook for human review when it writes
# somewhere but the path is computed at runtime (e.g., df.write_delta(path)).
# Hits land in the output table as rows with direction='potential_write' and
# the call line in `match_snippet` (url=None, secret_type=None).
WRITE_CALL_RE = re.compile(
    r"(?:"
    r"\.(?:write|to)_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|feather|html|sql|orc|hdf|pickle|stata|xml|database|table|gbq)\s*\("
    r"|\.write(?:Stream)?\.(?:format|mode|option|partitionBy|bucketBy|trigger|outputMode|start|save|saveAsTable|insertInto|csv|parquet|json|orc|text|delta|jdbc)\s*\("
    r"|\b(?:requests|httpx|aiohttp|session|client|http)\.(?:post|put|patch|delete)\s*\("
    r"|\b(?:upload_blob|upload_file|upload_fileobj|upload_from_string|upload_from_file|upload_data|put_object|put_block|put_block_blob|put_item|put_record|create_blob_from_\w+)\s*\("
    r"|\bmssparkutils\.fs\.(?:cp|put|append|mv)\s*\("
    r"|\.execute\s*\(\s*[\'\"](?:INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO|TRUNCATE\s+|DROP\s+|CREATE\s+TABLE|ALTER\s+TABLE)"
    r"|\bspark\.sql\s*\(\s*f?[\'\"](?:INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO|TRUNCATE\s+|DROP\s+TABLE|CREATE\s+TABLE|REPLACE\s+TABLE|ALTER\s+TABLE)"
    r"|\bopen\s*\([^)]*[\'\"](?:w|wb|w\+|wb\+|a|ab|a\+|ab\+|x|xb|x\+|xb\+|r\+|rb\+)[\'\"]"
    r"|!\s*curl[^\n]*-X\s+(?:POST|PUT|PATCH|DELETE)"
    r"|!\s*curl[^\n]*(?:--upload-file|--data\b|-d\s)"
    r")",
    re.IGNORECASE,
)
SECRET_PATTERNS = [
    ("Storage Account Key",
     re.compile(r'AccountKey\s*=\s*[A-Za-z0-9+/=]{40,}', re.IGNORECASE)),
    ("SAS Token",
     re.compile(r'(\?|&)sig=[A-Za-z0-9%+/=]{20,}', re.IGNORECASE)),
    ("SAS Token (assigned)",
     re.compile(r'["\']https?://[^"\s]+\.blob\.core\.windows\.net[^"\s]*sig=[^"\s]+["\']',
                re.IGNORECASE)),
    ("SQL Password",
     re.compile(r'(Password|pwd)\s*=\s*[^;\s"]{4,}', re.IGNORECASE)),
    ("Cosmos DB Key",
     re.compile(r'(AccountKey|cosmosdb[_-]?key)\s*[=:]\s*["\']?[A-Za-z0-9+/=]{40,}',
                re.IGNORECASE)),
    ("API Key",
     re.compile(r'(api[_-]?key|apikey|subscription[_-]?key|Ocp-Apim-Subscription-Key)\s*[=:]\s*["\']?[A-Za-z0-9]{16,}',
                re.IGNORECASE)),
    ("Client Secret",
     re.compile(r'(client[_-]?secret|app[_-]?secret)\s*[=:]\s*["\']?[A-Za-z0-9~._-]{16,}',
                re.IGNORECASE)),
    ("Connection String",
     re.compile(r'(connection[_-]?string|conn[_-]?str)\s*[=:]\s*["\'][^"]{20,}["\']',
                re.IGNORECASE)),
    ("Hardcoded Bearer Token",
     re.compile(r'["\']Bearer\s+eyJ[A-Za-z0-9_-]{20,}', re.IGNORECASE)),
    ("Event Hub / Service Bus Key",
     re.compile(r'SharedAccessKey\s*=\s*[A-Za-z0-9+/=]{30,}', re.IGNORECASE)),
    ("OpenAI API Key",
     re.compile(r'(openai[_-]?api[_-]?key|OPENAI_API_KEY)\s*[=:]\s*["\']?sk-[A-Za-z0-9]{20,}',
                re.IGNORECASE)),
    ("Azure OpenAI Key",
     re.compile(r'(azure[_-]?openai[_-]?key|AZURE_OPENAI_KEY)\s*[=:]\s*["\']?[A-Fa-f0-9]{32}',
                re.IGNORECASE)),
    ("Generic Secret Assignment",
     re.compile(r'(secret|password|passwd|token|credential)\s*[=:]\s*["\'][^"\s]{8,}["\']',
                re.IGNORECASE)),
    ("Spark Config Account Key",
     re.compile(r'spark\.conf\.set\s*\([^)]*account\.key[^)]*\)', re.IGNORECASE)),
    ("JDBC Password",
     re.compile(r'jdbc:[a-z]+://[^\s]*password=[^;\s"]{4,}', re.IGNORECASE)),
]


def _clean(u: str) -> str:
    while u and u[-1] in TRAILING:
        u = u[:-1]
    return u


def _redact(text: str, head: int = 4, tail: int = 4) -> str:
    """Mask the middle of a matched secret. Keeps head + tail chars so the
    finding is identifiable across scans without exposing the full value."""
    n = len(text)
    if n <= head + tail + 3:
        return "*" * n
    return text[:head] + ("*" * (n - head - tail)) + text[-tail:]


def extract_urls(text):
    """Yields (url, start_offset, end_offset). Offsets index into `text`."""
    seen, out = set(), []
    for m in URL_RE.finditer(text):
        s = m.start()
        u = _clean(m.group(0))
        if not u or u in seen:
            continue
        seen.add(u)
        out.append((u, s, s + len(u)))
    return out


def find_secrets(text):
    """Returns deduped list of (secret_type, redacted_snippet) for `text`."""
    seen, out = set(), []
    for name, rx in SECRET_PATTERNS:
        for m in rx.finditer(text):
            snip = _redact(m.group(0))
            key = (name, snip)
            if key in seen:
                continue
            seen.add(key)
            out.append((name, snip))
    return out


def find_write_calls(text):
    """Notebook-level write-call detection (destination-agnostic).

    Returns deduped list of (call_signature, line_snippet) tuples. The
    presence of any hit means the notebook DOES write to something — useful
    for flagging notebooks whose destinations are runtime-constructed
    (e.g., df.write_delta(path, ...)).
    """
    if not text:
        return []
    seen, out = set(), []
    for m in WRITE_CALL_RE.finditer(text):
        s = m.start()
        line_start = text.rfind("\n", 0, s) + 1
        line_end = text.find("\n", m.end())
        if line_end == -1:
            line_end = len(text)
        line = text[line_start:line_end].strip()
        if len(line) > 200:
            line = line[:200] + "..."
        sig = m.group(0).strip()
        key = (sig.lower(), line)
        if key in seen:
            continue
        seen.add(key)
        out.append((sig, line))
    return out


def classify_direction(text, url_start, window=200):
    """Classify a URL match against the I/O patterns in the text BEFORE it.

    Returns one of: 'read', 'write', 'read_write', 'unknown'.
    The closer match (to the URL) wins; ties become 'read_write'.
    """
    ws = max(0, url_start - window)
    pre = text[ws:url_start]
    r_pos = w_pos = -1
    for m in READ_RE.finditer(pre):
        r_pos = m.start()
    for m in WRITE_RE.finditer(pre):
        w_pos = m.start()
    if r_pos >= 0 and w_pos >= 0:
        if r_pos > w_pos:
            return "read"
        if w_pos > r_pos:
            return "write"
        return "read_write"
    if r_pos >= 0:
        return "read"
    if w_pos >= 0:
        return "write"
    return "unknown"


# Captures workspace identifier from OneLake ABFSS, Fabric REST, and Power BI
# URLs. Group order: 1=OneLake container, 2=Fabric REST guid, 3=Power BI guid.
WORKSPACE_URL_RE = re.compile(
    r"(?:"
    r"abfss?://([^@/\s]+)@onelake\.dfs\.fabric\.microsoft\.com"
    r"|https?://[^/\s]*\.fabric\.microsoft\.com/(?:[^?\s]*?/)?(?:v1/)?workspaces/([0-9a-f-]{8,})"
    r"|https?://(?:app|api)\.powerbi\.com/(?:[^/\s]*/)*?groups/([0-9a-f-]{8,})"
    r")",
    re.IGNORECASE,
)

# Lowercase 8-4-4-4-12 hex GUID with optional dashes.
GUID_RE = re.compile(
    r"^[0-9a-f]{8}-?[0-9a-f]{4}-?[0-9a-f]{4}-?[0-9a-f]{4}-?[0-9a-f]{12}$",
    re.IGNORECASE,
)


def parse_dest_workspace(url):
    """Extract the destination workspace identifier from a URL, or None.

    Returns whatever's in the URL — could be a GUID or a workspace name. Use
    resolve_dest_workspace() to normalize to (id, name).
    """
    if not url:
        return None
    m = WORKSPACE_URL_RE.search(url)
    if not m:
        return None
    return m.group(1) or m.group(2) or m.group(3)


def resolve_dest_workspace(url, src_ws_id, src_ws_name,
                           ws_name_by_id=None, ws_id_by_name=None):
    """Returns (dest_id, dest_name, cross_workspace_bool_or_None).

    cross_workspace is True iff dest != source AND we could resolve dest;
    False iff dest == source; None if dest is unknown (no parse) or source
    isn't a GUID we can compare against.
    """
    raw = parse_dest_workspace(url)
    if not raw:
        return None, None, None
    name_map = ws_name_by_id or {}
    id_map = ws_id_by_name or {}
    if GUID_RE.match(raw):
        dest_id = raw.lower()
        dest_name = name_map.get(dest_id, "") or name_map.get(raw, "")
    else:
        dest_name = raw
        dest_id = id_map.get(raw.lower(), "")
    # Compare. If we have neither id nor name from the source, we can't tell.
    src_id_l = (src_ws_id or "").lower()
    src_name_l = (src_ws_name or "").lower()
    if dest_id and src_id_l and GUID_RE.match(src_id_l or ""):
        cross = (dest_id != src_id_l)
    elif dest_name and src_name_l:
        cross = (dest_name.lower() != src_name_l)
    else:
        cross = None
    return (dest_id or None), (dest_name or None), cross


In [ ]:
# === Source diagnostics ======================================================
# In API mode, probe each enumeration endpoint with the current token.
# In Lakehouse mode, probe the source path to confirm it exists and matches.

if SOURCE_MODE == "lakehouse":
    print(f"[Lakehouse mode] Source: {source_uri}  (glob: {SOURCE_FILE_GLOB})")
    try:
        try:
            import notebookutils as _nbu
            _ls = _nbu.fs.ls(source_uri)
        except Exception:
            import mssparkutils as _nbu  # noqa: F401
            _ls = _nbu.fs.ls(source_uri)
        files = [e for e in _ls if not getattr(e, "isDir", False)]
        dirs  = [e for e in _ls if getattr(e, "isDir", False)]
        print(f"  Top-level entries: {len(_ls)}  "
              f"({len(files)} files, {len(dirs)} folders)")
        for e in list(_ls)[:5]:
            kind = "DIR " if getattr(e, "isDir", False) else "FILE"
            print(f"    {kind}  {getattr(e, 'name', e)}")
        print("  Done. Empty result here usually means a wrong path or the "
              "identity lacks Files.Read on this Lakehouse.")
    except Exception as e:
        print(f"  [ERR] Could not list {source_uri}: "
              f"{type(e).__name__}: {e}")
else:
    import requests as _req
    _diag_headers = {"Authorization": f"Bearer {_token}",
                     "Content-Type": "application/json"}

    def _probe(label, method, url, **kw):
        try:
            r = _req.request(method, url, headers=_diag_headers,
                             timeout=30, **kw)
            sample = ""
            try:
                j = r.json()
                if isinstance(j, dict):
                    if "value" in j and isinstance(j["value"], list):
                        sample = f"value[]: {len(j['value'])} rows"
                    else:
                        sample = ", ".join(list(j.keys())[:6])
                else:
                    sample = str(j)[:120]
            except Exception:
                sample = r.text[:120]
            print(f"  [{r.status_code}] {label:34s}  {sample}")
            return r.status_code
        except Exception as e:
            print(f"  [ERR] {label:34s}  {type(e).__name__}: {e}")
            return None

    print("Probing endpoints with current identity ...")
    _probe("PBI admin /myorg/admin/groups",
           "GET", f"{PBI_BASE}/v1.0/myorg/admin/groups",
           params={"$top": 1})
    _probe("Fabric admin /v1/admin/workspaces",
           "GET", f"{FABRIC_BASE}/v1/admin/workspaces")
    _probe("Fabric admin /v1/admin/items",
           "GET", f"{FABRIC_BASE}/v1/admin/items")
    _probe("Fabric user /v1/workspaces",
           "GET", f"{FABRIC_BASE}/v1/workspaces")
    print("Done. 200 = accessible; 401/403 = identity lacks that role; "
          "404 = endpoint path wrong.")


In [ ]:
# === Enumerate workspaces + notebooks ========================================
# Strategy:
#   1. List workspaces.  Try in order until one returns >0 rows:
#         a) PBI admin /v1.0/myorg/admin/groups      (legacy, stable)
#         b) Fabric admin /v1/admin/workspaces       (no type filter)
#         c) Fabric /v1/workspaces                   (user / SP membership)
#   2. List notebooks per workspace, in parallel, using whichever path matches
#      the caller's role (admin items endpoint vs. user items endpoint).
# Every step prints HTTP status + counts so you can see exactly where it stops.


async def _http_json(session, method, url, **kw):
    """One request. Returns (status, body_json_or_text, headers). Never raises
    for HTTP-level errors so the caller can decide whether to fall back."""
    async with session.request(method, url, **kw) as r:
        try:
            body = await r.json(content_type=None)
        except Exception:
            body = await r.text()
        return r.status, body, dict(r.headers)


async def _list_pbi_admin_workspaces(session):
    """GET https://api.powerbi.com/v1.0/myorg/admin/groups?$top=5000&$skip=N"""
    out, skip, page_size = [], 0, 5000
    while True:
        params = {"$top": page_size, "$skip": skip}
        status, body, _ = await _http_json(
            session, "GET",
            f"{PBI_BASE}/v1.0/myorg/admin/groups",
            params=params,
        )
        if status != 200:
            raise RuntimeError(f"PBI admin/groups HTTP {status}: {str(body)[:300]}")
        page = body.get("value", []) if isinstance(body, dict) else []
        out.extend(page)
        if len(page) < page_size:
            break
        skip += page_size
    return out


async def _list_fabric_admin_workspaces(session):
    """GET https://api.fabric.microsoft.com/v1/admin/workspaces  (no filters)"""
    out, url = [], f"{FABRIC_BASE}/v1/admin/workspaces"
    while url:
        status, body, _ = await _http_json(session, "GET", url)
        if status != 200:
            raise RuntimeError(f"Fabric admin/workspaces HTTP {status}: {str(body)[:300]}")
        out.extend((body or {}).get("value", []))
        url = (body or {}).get("continuationUri")
    return out


async def _list_user_workspaces(session):
    """GET https://api.fabric.microsoft.com/v1/workspaces"""
    out, url = [], f"{FABRIC_BASE}/v1/workspaces"
    while url:
        status, body, _ = await _http_json(session, "GET", url)
        if status != 200:
            raise RuntimeError(f"User /workspaces HTTP {status}: {str(body)[:300]}")
        out.extend((body or {}).get("value", []))
        url = (body or {}).get("continuationUri")
    return out


async def _list_admin_items_tenant(session):
    """GET /v1/admin/items — tenant-wide items, paged. Filter notebook client-side."""
    out, url = [], f"{FABRIC_BASE}/v1/admin/items"
    first_sample = None
    while url:
        status, body, _ = await _http_json(session, "GET", url)
        if status != 200:
            raise RuntimeError(f"Fabric admin/items HTTP {status}: {str(body)[:300]}")
        items = (body or {}).get("itemEntities") or (body or {}).get("value") or []
        if first_sample is None and items:
            first_sample = items[:3]
        for it in items:
            it_type = (it.get("type") or it.get("itemType") or "").lower()
            if it_type == "notebook":
                ws_id = (it.get("workspaceId")
                         or (it.get("workspace") or {}).get("id"))
                if ws_id:
                    out.append({"workspaceId": ws_id, "id": it["id"],
                                "displayName": it.get("displayName") or it.get("name")})
        url = (body or {}).get("continuationUri")
    return out, first_sample


async def _list_admin_items_workspace(session, wid):
    """GET /v1/admin/items?workspaceId={wid} — per-workspace, paged."""
    out, url = [], f"{FABRIC_BASE}/v1/admin/items?workspaceId={wid}"
    first_sample = None
    while url:
        status, body, _ = await _http_json(session, "GET", url)
        if status != 200:
            return None, status, None
        items = (body or {}).get("itemEntities") or (body or {}).get("value") or []
        if first_sample is None and items:
            first_sample = items[:3]
        for it in items:
            it_type = (it.get("type") or it.get("itemType") or "").lower()
            if it_type == "notebook":
                out.append({"workspaceId": wid, "id": it["id"],
                            "displayName": it.get("displayName") or it.get("name")})
        url = (body or {}).get("continuationUri")
    return out, 200, first_sample


async def _list_user_items_workspace(session, wid):
    """GET /v1/workspaces/{wid}/items — user items, paged, filter client-side."""
    out, url = [], f"{FABRIC_BASE}/v1/workspaces/{wid}/items"
    first_sample = None
    while url:
        status, body, _ = await _http_json(session, "GET", url)
        if status != 200:
            return None, status, None
        items = (body or {}).get("value", [])
        if first_sample is None and items:
            first_sample = items[:3]
        for it in items:
            it_type = (it.get("type") or "").lower()
            if it_type == "notebook":
                out.append({"workspaceId": wid, "id": it["id"],
                            "displayName": it.get("displayName")})
        url = (body or {}).get("continuationUri")
    return out, 200, first_sample


async def run_enumeration(token):
    headers = {"Authorization": f"Bearer {token}",
               "Content-Type": "application/json"}
    timeout = aiohttp.ClientTimeout(total=900, connect=30)
    async with aiohttp.ClientSession(headers=headers, timeout=timeout) as session:

        # ---- Step 1: list workspaces ----------------------------------------
        used_admin = False
        workspaces = []
        if ADMIN_MODE:
            chain = [
                ("PBI admin groups",       _list_pbi_admin_workspaces, True),
                ("Fabric admin workspaces", _list_fabric_admin_workspaces, True),
                ("User /v1/workspaces",    _list_user_workspaces, False),
            ]
        else:
            chain = [("User /v1/workspaces", _list_user_workspaces, False)]

        for label, fn, is_admin in chain:
            try:
                ws = await fn(session)
                print(f"  [{label}] returned {len(ws)} workspaces.")
                if ws:
                    workspaces = ws
                    used_admin = is_admin
                    break
            except Exception as e:
                print(f"  [{label}] FAILED: {e}")

        ws_ids = [w["id"] for w in workspaces]
        # Build a stable id -> name map so each notebook can carry its
        # workspace's human-readable name through to the output table.
        ws_name_by_id = {
            w["id"]: (w.get("name") or w.get("displayName") or w["id"])
            for w in workspaces
        }
        if READ_WORKSPACE_IDS:
            allow = set(READ_WORKSPACE_IDS)
            ws_ids = [w for w in ws_ids if w in allow]
            print(f"  Applied allowlist -> {len(ws_ids)} workspace(s).")

        if not ws_ids:
            print("  No workspaces resolved. Check permissions (Fabric admin "
                  "role + 'SPs can call read-only admin APIs' tenant setting "
                  "if running as service principal).")
            return []

        # Quick visibility: show a few workspace names so the caller can sanity-
        # check that the tenant being scanned is the expected one.
        sample = workspaces[:5]
        print("  Sample workspaces:",
              ", ".join((w.get("name") or w.get("displayName") or w["id"])
                        for w in sample))

        # ---- Step 2: list notebooks ----------------------------------------
        notebooks = []

        if used_admin:
            # Tenant-wide first: one paged stream of every item in the tenant.
            # Filter notebooks (and optionally the workspace allowlist) client-
            # side. Much faster than N per-workspace calls.
            print("  Trying tenant-wide /v1/admin/items (single paged stream)...")
            try:
                tenant_nbs, first_sample = await _list_admin_items_tenant(session)
                print(f"  Tenant-wide returned {len(tenant_nbs)} notebooks.")
                if READ_WORKSPACE_IDS:
                    allow = set(READ_WORKSPACE_IDS)
                    tenant_nbs = [n for n in tenant_nbs
                                  if n["workspaceId"] in allow]
                    print(f"  After allowlist: {len(tenant_nbs)} notebooks.")
                if tenant_nbs:
                    for n in tenant_nbs:
                        n["workspaceName"] = ws_name_by_id.get(
                            n.get("workspaceId"), n.get("workspaceId", ""))
                    return tenant_nbs
                if first_sample:
                    print("  Tenant-wide saw items but none were notebooks. "
                          "Sample item shapes:")
                    import json as _json
                    for it in first_sample:
                        snippet = {k: it.get(k) for k in
                                   ("id", "type", "itemType", "displayName", "name",
                                    "workspaceId")}
                        print(f"    {_json.dumps(snippet)}")
                print("  Falling back to per-workspace /v1/admin/items ...")
            except Exception as e:
                print(f"  Tenant-wide FAILED: {e}; "
                      "falling back to per-workspace /v1/admin/items ...")

            sem = asyncio.Semaphore(50)

            async def one_admin(wid):
                async with sem:
                    return wid, await _list_admin_items_workspace(session, wid)

            results = await asyncio.gather(*(one_admin(w) for w in ws_ids))
        else:
            sem = asyncio.Semaphore(50)

            async def one_user(wid):
                async with sem:
                    return wid, await _list_user_items_workspace(session, wid)

            results = await asyncio.gather(*(one_user(w) for w in ws_ids))

        status_counts = {}
        first_nonempty_sample = None
        for wid, (items, status, sample) in results:
            status_counts[status] = status_counts.get(status, 0) + 1
            if items:
                notebooks.extend(items)
            elif sample and first_nonempty_sample is None:
                first_nonempty_sample = (wid, sample)

        print(f"  Per-workspace item listing status codes: {status_counts}")

        if not notebooks and first_nonempty_sample is not None:
            wid, sample = first_nonempty_sample
            print(f"\n  DIAGNOSTIC: workspace {wid} returned items but no "
                  f"items had type=='notebook'. Sample item shapes:")
            import json as _json
            for it in sample:
                snippet = {k: it.get(k) for k in
                           ("id", "type", "itemType", "displayName", "name")}
                print(f"    keys={list(it.keys())[:12]}")
                print(f"    {_json.dumps(snippet)}")
            print("  If 'type' uses a different label (e.g., 'SynapseNotebook'),")
            print("  add it to _item_to_notebook_row()'s match list.")

        # Stamp human-readable workspace name onto each notebook before
        # returning so it flows through the inventory + findings tables.
        for n in notebooks:
            n["workspaceName"] = ws_name_by_id.get(
                n.get("workspaceId"), n.get("workspaceId", ""))
        return notebooks


def _run_async(coro):
    """Run an asyncio coroutine from a Jupyter/Fabric notebook safely.

    The Fabric notebook kernel keeps its own event loop running, so calling
    loop.run_until_complete() on the main thread raises 'cannot run the event
    loop while another loop is running'. Running the coroutine on a worker
    thread gives it a fresh, isolated loop.
    """
    import threading

    try:
        asyncio.get_running_loop()
        running = True
    except RuntimeError:
        running = False

    if not running:
        return asyncio.run(coro)

    box = {}

    def worker():
        new_loop = asyncio.new_event_loop()
        try:
            asyncio.set_event_loop(new_loop)
            box["value"] = new_loop.run_until_complete(coro)
        except BaseException as e:
            box["error"] = e
        finally:
            new_loop.close()

    t = threading.Thread(target=worker, name="fabric-scanner-asyncio")
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box["value"]


notebooks = [] if SOURCE_MODE == "lakehouse" \
                else _run_async(run_enumeration(_token))

if MAX_NOTEBOOKS and SOURCE_MODE == "api":
    notebooks = notebooks[:MAX_NOTEBOOKS]

# Build workspace lookup maps for cross-workspace detection. Each notebook
# entry already carries (workspaceId, workspaceName); derive bidirectional
# maps from that. In lakehouse mode the API list is empty, so seed it with
# the configured source workspace (if any).
_WS_NAME_BY_ID = {
    n["workspaceId"]: (n.get("workspaceName") or n["workspaceId"])
    for n in notebooks
    if n.get("workspaceId")
}
if SOURCE_MODE == "lakehouse" and SOURCE_LAKEHOUSE_WORKSPACE_ID:
    _WS_NAME_BY_ID[SOURCE_LAKEHOUSE_WORKSPACE_ID] = (
        SOURCE_LAKEHOUSE_WORKSPACE_NAME or SOURCE_LAKEHOUSE_WORKSPACE_ID
    )
_WS_NAME_BY_ID = {k.lower(): v for k, v in _WS_NAME_BY_ID.items()}
_WS_ID_BY_NAME = {v.lower(): k for k, v in _WS_NAME_BY_ID.items() if v}
print(f"  Workspace map: {len(_WS_NAME_BY_ID)} entries available "
      f"for cross-workspace lookup.")

if SOURCE_MODE == "lakehouse":
    print(f"[Lakehouse mode] Skipping API enumeration. Files will be "
          f"discovered by the next cell from: {source_uri}")
else:
    print(f"\nInventory: {len(notebooks)} notebooks total.")


In [ ]:
# === Build inventory DataFrame + partition ===================================
if SOURCE_MODE == "lakehouse":
    from pyspark.sql import functions as F
    file_df = (spark.read.format("binaryFile")
                 .option("recursiveFileLookup", "true")
                 .option("pathGlobFilter", SOURCE_FILE_GLOB)
                 .load(source_uri))
    n_total = file_df.count()
    if n_total == 0:
        raise SystemExit(
            f"No files matched glob '{SOURCE_FILE_GLOB}' under '{source_uri}'."
        )

    # Optional smoke-test cap.
    if MAX_NOTEBOOKS:
        file_df = file_df.limit(MAX_NOTEBOOKS)
        n_total = min(n_total, MAX_NOTEBOOKS)

    # Inventory snapshot (no content bytes) — written to the inventory table.
    snapshot = file_df.select(
        F.lit(SOURCE_LAKEHOUSE_WORKSPACE_ID or "lakehouse").alias("workspace_id"),
        F.lit(SOURCE_LAKEHOUSE_WORKSPACE_NAME
              or SOURCE_LAKEHOUSE_WORKSPACE_ID
              or "lakehouse").alias("workspace_name"),
        F.col("path").alias("notebook_id"),
        F.regexp_extract(F.col("path"), "([^/]+)$", 1).alias("display_name"),
    )

    if WRITE_TO_DEFAULT_LAKEHOUSE:
        snapshot.write.mode("overwrite").saveAsTable(inventory_target)
    else:
        (snapshot.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(inventory_target))

    num_partitions = max(1, (n_total + TARGET_PARTITION_SIZE - 1)
                              // TARGET_PARTITION_SIZE)
    # inventory_df is the *scan input* — keep `path` + `content` columns.
    inventory_df = file_df.repartition(num_partitions)
    print(f"{n_total} files matched.  Scanning in {num_partitions} partitions.")
    print(f"Inventory persisted to: {inventory_target}")

else:
    if not notebooks:
        raise SystemExit("No notebooks found — nothing to scan.")

    inventory_rows = [
        Row(workspace_id=n["workspaceId"],
            workspace_name=(n.get("workspaceName") or n["workspaceId"]),
            notebook_id=n["id"],
            display_name=(n.get("displayName") or ""))
        for n in notebooks
    ]
    inventory_df = spark.createDataFrame(inventory_rows)

    if WRITE_TO_DEFAULT_LAKEHOUSE:
        inventory_df.write.mode("overwrite").saveAsTable(inventory_target)
    else:
        (inventory_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(inventory_target))

    n_total = inventory_df.count()
    num_partitions = max(1, (n_total + TARGET_PARTITION_SIZE - 1)
                              // TARGET_PARTITION_SIZE)
    inventory_df = inventory_df.repartition(num_partitions)
    print(f"{n_total} notebooks across {num_partitions} partitions "
          f"(~{TARGET_PARTITION_SIZE} per partition).")
    print(f"Inventory persisted to: {inventory_target}")


In [ ]:
# === Fan out getDefinition with mapPartitions ================================

_FABRIC_BASE_BC = FABRIC_BASE
_CONC_BC = EXECUTOR_CONCURRENCY
_WS_NAME_BY_ID_BC = _WS_NAME_BY_ID
_WS_ID_BY_NAME_BC = _WS_ID_BY_NAME
_MAX_RETRIES_BC = MAX_RETRIES
_AUDIENCE_BC = TOKEN_AUDIENCE
_FALLBACK_TOKEN_BC = _token   # fallback if executor cannot refresh


def fetch_partition(rows):
    """Runs on each Spark executor. Async-fetches every notebook in this
    partition, scans for URLs, and yields Row objects."""
    import asyncio
    import aiohttp
    import base64
    import json
    import re
    from datetime import datetime, timezone
    from pyspark.sql import Row

    rows = list(rows)
    if not rows:
        return iter([])

    # Try to refresh token on the executor so very long jobs don't expire.
    try:
        import notebookutils
        token = notebookutils.credentials.getToken(_AUDIENCE_BC)
    except Exception:
        try:
            import mssparkutils
            token = mssparkutils.credentials.getToken(_AUDIENCE_BC)
        except Exception:
            token = _FALLBACK_TOKEN_BC

    URL_RE = re.compile(
        r"(?i)\b(?:https?|ftps?|file|abfss?|wasbs?|s3a?|s3n|gs|dbfs|adl|hdfs)://"
        r"[^\s'\"<>`)\]}]+"
        r"|\bwww\.[^\s'\"<>`)\]}]+\.[^\s'\"<>`)\]}]+"
    )
    TRAILING = ".,;:!?)]}>\"'`"

    READ_RE = re.compile(
        r"(?:"
        r"\brequests?\.(?:get|head|options)\s*\("
        r"|\burllib(?:\.request)?\.urlopen\s*\("
        r"|\burlretrieve\s*\("
        r"|\b(?:pd|pandas|pl|polars)\.(?:read|scan)_\w+\s*\("
        r"|\.read_\w+\s*\("
        r"|\.scan_(?:parquet|csv|ndjson|delta|ipc|arrow)\s*\("
        r"|\bspark\.read\b"
        r"|\b(?:httpx|aiohttp)\.(?:get|head|options)\s*\("
        r"|\bsession\.(?:get|head|options)\s*\("
        r"|\bfetch\s*\("
        r"|\bsc\.textFile\s*\("
        r"|\bspark\.sparkContext\.textFile\s*\("
        r"|!\s*wget\b"
        r"|!\s*curl\b(?![^\n]*-X\s+(?:POST|PUT|PATCH|DELETE))"
        r"|\bdownload\w*\s*\("
        r")",
        re.IGNORECASE,
    )

    WRITE_RE = re.compile(
        r"(?:"
        r"\brequests?\.(?:post|put|patch|delete)\s*\("
        r"|\b(?:httpx|aiohttp)\.(?:post|put|patch|delete)\s*\("
        r"|\bsession\.(?:post|put|patch|delete)\s*\("
        r"|\b(?:pd|pandas)\.to_\w+\s*\("
        r"|\.to_(?:csv|json|parquet|sql|excel|html|hdf|pickle|orc|feather)\s*\("
        r"|\.write_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|database)\s*\("
        r"|\.write\.\w+"
        r"|\.save(?:AsTable|AsTextFile)?\s*\("
        r"|\bupload\w*\s*\("
        r"|\bput_object\s*\("
        r"|!\s*curl[^\n]*-X\s+(?:POST|PUT|PATCH|DELETE)"
        r"|!\s*curl[^\n]*(?:--upload-file|--data\b|-d\s)"
        r")",
        re.IGNORECASE,
    )

    WRITE_CALL_RE = re.compile(
        r"(?:"
        r"\.(?:write|to)_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|feather|html|sql|orc|hdf|pickle|stata|xml|database|table|gbq)\s*\("
        r"|\.write(?:Stream)?\.(?:format|mode|option|partitionBy|bucketBy|trigger|outputMode|start|save|saveAsTable|insertInto|csv|parquet|json|orc|text|delta|jdbc)\s*\("
        r"|\b(?:requests|httpx|aiohttp|session|client|http)\.(?:post|put|patch|delete)\s*\("
        r"|\b(?:upload_blob|upload_file|upload_fileobj|upload_from_string|upload_from_file|upload_data|put_object|put_block|put_block_blob|put_item|put_record|create_blob_from_\w+)\s*\("
        r"|\bmssparkutils\.fs\.(?:cp|put|append|mv)\s*\("
        r"|\.execute\s*\(\s*[\'\"](?:INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO|TRUNCATE\s+|DROP\s+|CREATE\s+TABLE|ALTER\s+TABLE)"
        r"|\bspark\.sql\s*\(\s*f?[\'\"](?:INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO|TRUNCATE\s+|DROP\s+TABLE|CREATE\s+TABLE|REPLACE\s+TABLE|ALTER\s+TABLE)"
        r"|\bopen\s*\([^)]*[\'\"](?:w|wb|w\+|wb\+|a|ab|a\+|ab\+|x|xb|x\+|xb\+|r\+|rb\+)[\'\"]"
        r"|!\s*curl[^\n]*-X\s+(?:POST|PUT|PATCH|DELETE)"
        r"|!\s*curl[^\n]*(?:--upload-file|--data\b|-d\s)"
        r")",
        re.IGNORECASE,
    )

    SECRET_PATTERNS = [
        ("Storage Account Key",
         re.compile(r'AccountKey\s*=\s*[A-Za-z0-9+/=]{40,}', re.IGNORECASE)),
        ("SAS Token",
         re.compile(r'(\?|&)sig=[A-Za-z0-9%+/=]{20,}', re.IGNORECASE)),
        ("SAS Token (assigned)",
         re.compile(r'["\']https?://[^"\s]+\.blob\.core\.windows\.net[^"\s]*sig=[^"\s]+["\']',
                    re.IGNORECASE)),
        ("SQL Password",
         re.compile(r'(Password|pwd)\s*=\s*[^;\s"]{4,}', re.IGNORECASE)),
        ("Cosmos DB Key",
         re.compile(r'(AccountKey|cosmosdb[_-]?key)\s*[=:]\s*["\']?[A-Za-z0-9+/=]{40,}',
                    re.IGNORECASE)),
        ("API Key",
         re.compile(r'(api[_-]?key|apikey|subscription[_-]?key|Ocp-Apim-Subscription-Key)\s*[=:]\s*["\']?[A-Za-z0-9]{16,}',
                    re.IGNORECASE)),
        ("Client Secret",
         re.compile(r'(client[_-]?secret|app[_-]?secret)\s*[=:]\s*["\']?[A-Za-z0-9~._-]{16,}',
                    re.IGNORECASE)),
        ("Connection String",
         re.compile(r'(connection[_-]?string|conn[_-]?str)\s*[=:]\s*["\'][^"]{20,}["\']',
                    re.IGNORECASE)),
        ("Hardcoded Bearer Token",
         re.compile(r'["\']Bearer\s+eyJ[A-Za-z0-9_-]{20,}', re.IGNORECASE)),
        ("Event Hub / Service Bus Key",
         re.compile(r'SharedAccessKey\s*=\s*[A-Za-z0-9+/=]{30,}', re.IGNORECASE)),
        ("OpenAI API Key",
         re.compile(r'(openai[_-]?api[_-]?key|OPENAI_API_KEY)\s*[=:]\s*["\']?sk-[A-Za-z0-9]{20,}',
                    re.IGNORECASE)),
        ("Azure OpenAI Key",
         re.compile(r'(azure[_-]?openai[_-]?key|AZURE_OPENAI_KEY)\s*[=:]\s*["\']?[A-Fa-f0-9]{32}',
                    re.IGNORECASE)),
        ("Generic Secret Assignment",
         re.compile(r'(secret|password|passwd|token|credential)\s*[=:]\s*["\'][^"\s]{8,}["\']',
                    re.IGNORECASE)),
        ("Spark Config Account Key",
         re.compile(r'spark\.conf\.set\s*\([^)]*account\.key[^)]*\)', re.IGNORECASE)),
        ("JDBC Password",
         re.compile(r'jdbc:[a-z]+://[^\s]*password=[^;\s"]{4,}', re.IGNORECASE)),
    ]

    def _clean(u):
        while u and u[-1] in TRAILING:
            u = u[:-1]
        return u

    def _redact(text, head=4, tail=4):
        n = len(text)
        if n <= head + tail + 3:
            return "*" * n
        return text[:head] + ("*" * (n - head - tail)) + text[-tail:]

    def _extract(text):
        """Yields (url, start_offset, end_offset) within `text`."""
        seen, out = set(), []
        for m in URL_RE.finditer(text):
            s = m.start()
            u = _clean(m.group(0))
            if not u or u in seen:
                continue
            seen.add(u)
            out.append((u, s, s + len(u)))
        return out

    def _find_secrets(text):
        seen, out = set(), []
        for name, rx in SECRET_PATTERNS:
            for m in rx.finditer(text):
                snip = _redact(m.group(0))
                key = (name, snip)
                if key in seen:
                    continue
                seen.add(key)
                out.append((name, snip))
        return out

    def _find_write_calls(text):
        """Destination-agnostic write-call detection. Returns deduped
        [(signature, line_snippet)] tuples.
        """
        if not text:
            return []
        seen, out = set(), []
        for m in WRITE_CALL_RE.finditer(text):
            s = m.start()
            line_start = text.rfind("\n", 0, s) + 1
            line_end = text.find("\n", m.end())
            if line_end == -1:
                line_end = len(text)
            line = text[line_start:line_end].strip()
            if len(line) > 200:
                line = line[:200] + "..."
            sig = m.group(0).strip()
            key = (sig.lower(), line)
            if key in seen:
                continue
            seen.add(key)
            out.append((sig, line))
        return out

    def _classify(text, url_start, window=200):
        """Classify a URL by the I/O call (if any) immediately preceding it."""
        ws = max(0, url_start - window)
        pre = text[ws:url_start]
        r_pos = w_pos = -1
        for m in READ_RE.finditer(pre):
            r_pos = m.start()
        for m in WRITE_RE.finditer(pre):
            w_pos = m.start()
        if r_pos >= 0 and w_pos >= 0:
            if r_pos > w_pos:
                return "read"
            if w_pos > r_pos:
                return "write"
            return "read_write"
        if r_pos >= 0:
            return "read"
        if w_pos >= 0:
            return "write"
        return "unknown"

    _WORKSPACE_URL_RE = re.compile(
        r"(?:"
        r"abfss?://([^@/\s]+)@onelake\.dfs\.fabric\.microsoft\.com"
        r"|https?://[^/\s]*\.fabric\.microsoft\.com/(?:[^?\s]*?/)?(?:v1/)?workspaces/([0-9a-f-]{8,})"
        r"|https?://(?:app|api)\.powerbi\.com/(?:[^/\s]*/)*?groups/([0-9a-f-]{8,})"
        r")",
        re.IGNORECASE,
    )
    _GUID_RE = re.compile(
        r"^[0-9a-f]{8}-?[0-9a-f]{4}-?[0-9a-f]{4}-?[0-9a-f]{4}-?[0-9a-f]{12}$",
        re.IGNORECASE,
    )
    _NAME_BY_ID = _WS_NAME_BY_ID_BC or {}
    _ID_BY_NAME = _WS_ID_BY_NAME_BC or {}

    def _resolve_dest(url, src_wid, src_wname):
        """Return (dest_id, dest_name, cross_workspace) tuple."""
        if not url:
            return None, None, None
        m = _WORKSPACE_URL_RE.search(url)
        if not m:
            return None, None, None
        raw = m.group(1) or m.group(2) or m.group(3)
        if not raw:
            return None, None, None
        if _GUID_RE.match(raw):
            dest_id = raw.lower()
            dest_name = _NAME_BY_ID.get(dest_id, "") or _NAME_BY_ID.get(raw, "")
        else:
            dest_name = raw
            dest_id = _ID_BY_NAME.get(raw.lower(), "")
        src_id_l = (src_wid or "").lower()
        src_name_l = (src_wname or "").lower()
        if dest_id and src_id_l and _GUID_RE.match(src_id_l):
            cross = (dest_id != src_id_l)
        elif dest_name and src_name_l:
            cross = (dest_name.lower() != src_name_l)
        else:
            cross = None
        return (dest_id or None), (dest_name or None), cross

    def _scan(definition):
        """Scan one notebook definition. Returns (url_hits, secret_hits, write_call_hits).

        url_hits:        list of (path, url, direction, source_kind)
        secret_hits:     list of (path, secret_type, snippet, source_kind)
        write_call_hits: list of (path, signature, line_snippet, source_kind)
        """
        url_hits = []
        secret_hits = []
        write_call_hits = []
        parts = (definition.get("definition") or {}).get("parts") or []
        for part in parts:
            path = part.get("path", "")
            if not (path.endswith(".py")
                    or path.endswith(".ipynb")
                    or path.endswith(".md")):
                continue
            try:
                raw = base64.b64decode(
                    part.get("payload", "")
                ).decode("utf-8", errors="ignore")
            except Exception:
                continue

            if path.endswith(".md"):
                for url, _s, _e in _extract(raw):
                    url_hits.append((path, url, "reference", "markdown"))
                for name, snip in _find_secrets(raw):
                    secret_hits.append((path, name, snip, "markdown"))
                continue

            if path.endswith(".py"):
                for url, s, _e in _extract(raw):
                    url_hits.append((path, url, _classify(raw, s), "code"))
                for name, snip in _find_secrets(raw):
                    secret_hits.append((path, name, snip, "code"))
                for sig, line in _find_write_calls(raw):
                    write_call_hits.append((path, sig, line, "code"))
                continue

            # .ipynb — classify per cell so markdown / outputs don't get
            # mislabeled as read/write.
            try:
                nb = json.loads(raw)
            except Exception:
                for url, s, _e in _extract(raw):
                    url_hits.append((path, url, "unknown", "code"))
                for name, snip in _find_secrets(raw):
                    secret_hits.append((path, name, snip, "code"))
                for sig, line in _find_write_calls(raw):
                    write_call_hits.append((path, sig, line, "code"))
                continue

            for cell in nb.get("cells", []) or []:
                ctype = cell.get("cell_type", "")
                src = cell.get("source", "")
                src_text = ("".join(src) if isinstance(src, list)
                            else str(src))

                if ctype == "markdown":
                    for url, _s, _e in _extract(src_text):
                        url_hits.append((path, url, "reference", "markdown"))
                    for name, snip in _find_secrets(src_text):
                        secret_hits.append((path, name, snip, "markdown"))
                    continue

                if ctype == "code":
                    for url, s, _e in _extract(src_text):
                        url_hits.append((path, url,
                                         _classify(src_text, s), "code"))
                    for name, snip in _find_secrets(src_text):
                        secret_hits.append((path, name, snip, "code"))
                    for sig, line in _find_write_calls(src_text):
                        write_call_hits.append((path, sig, line, "code"))
                    for out in cell.get("outputs", []) or []:
                        if not isinstance(out, dict):
                            continue
                        out_chunks = []
                        t = out.get("text", "")
                        out_chunks.append("".join(t) if isinstance(t, list)
                                          else str(t))
                        data = out.get("data") or {}
                        if isinstance(data, dict):
                            for mime, payload in data.items():
                                if mime.startswith(("text/", "application/")):
                                    out_chunks.append(
                                        "".join(payload)
                                        if isinstance(payload, list)
                                        else str(payload)
                                    )
                        out_text = "\n".join(out_chunks)
                        for url, _s, _e in _extract(out_text):
                            url_hits.append((path, url, "reference", "output"))
                        for name, snip in _find_secrets(out_text):
                            secret_hits.append((path, name, snip, "output"))
                    continue

                # raw / unknown cell type
                for url, _s, _e in _extract(src_text):
                    url_hits.append((path, url, "unknown", "code"))
                for name, snip in _find_secrets(src_text):
                    secret_hits.append((path, name, snip, "code"))
                for sig, line in _find_write_calls(src_text):
                    write_call_hits.append((path, sig, line, "code"))
        return url_hits, secret_hits, write_call_hits

    headers = {"Authorization": f"Bearer {token}"}
    sem = asyncio.Semaphore(_CONC_BC)

    async def fetch_one(session, wid, wname, iid, name):
        url = (f"{_FABRIC_BASE_BC}/v1/workspaces/{wid}/items/{iid}"
               f"/getDefinition?format=ipynb")
        async with sem:
            for attempt in range(_MAX_RETRIES_BC + 1):
                try:
                    async with session.post(url) as r:
                        if r.status == 200:
                            return wid, wname, iid, name, await r.json(), None
                        if r.status == 202:
                            poll_url = r.headers.get("Location")
                            if not poll_url:
                                return wid, wname, iid, name, None, "202 without Location"
                        elif r.status in (429, 500, 502, 503, 504):
                            await asyncio.sleep(
                                float(r.headers.get("Retry-After",
                                                    str(min(2 ** attempt, 60))))
                            )
                            continue
                        else:
                            txt = await r.text()
                            return wid, wname, iid, name, None, f"HTTP {r.status}: {txt[:200]}"
                    for _ in range(180):
                        await asyncio.sleep(2)
                        async with session.get(poll_url) as p:
                            if p.status == 200:
                                pj = await p.json()
                                if pj.get("status") == "Succeeded":
                                    async with session.get(poll_url + "/result") as res:
                                        if res.status == 200:
                                            return wid, wname, iid, name, await res.json(), None
                                        else:
                                            return wid, wname, iid, name, None, f"result HTTP {res.status}"
                                if pj.get("status") == "Failed":
                                    return wid, wname, iid, name, None, f"LRO failed: {pj}"
                            elif p.status == 429:
                                await asyncio.sleep(float(p.headers.get("Retry-After", "5")))
                    return wid, wname, iid, name, None, "LRO poll timeout"
                except Exception as e:
                    if attempt == _MAX_RETRIES_BC:
                        return wid, wname, iid, name, None, f"{type(e).__name__}: {e}"
                    await asyncio.sleep(min(2 ** attempt, 30))

    async def run_all():
        connector = aiohttp.TCPConnector(limit=_CONC_BC * 2)
        timeout = aiohttp.ClientTimeout(total=1200, connect=30)
        async with aiohttp.ClientSession(connector=connector,
                                         headers=headers,
                                         timeout=timeout) as session:
            tasks = [fetch_one(session, r.workspace_id, r.workspace_name,
                               r.notebook_id, r.display_name)
                     for r in rows]
            return await asyncio.gather(*tasks, return_exceptions=False)

    loop = asyncio.new_event_loop()
    try:
        results = loop.run_until_complete(run_all())
    finally:
        loop.close()

    now = datetime.now(timezone.utc)
    out_rows = []
    for wid, wname, iid, name, body, err in results:
        if err is not None or body is None:
            out_rows.append(Row(
                workspace_id=wid, workspace_name=wname,
                notebook_id=iid, display_name=name,
                part_path=None, url=None, direction=None, source_kind=None,
                secret_type=None, match_snippet=None,
                dest_workspace_id=None, dest_workspace_name=None,
                cross_workspace=None,
                error=err, scanned_at=now,
            ))
            continue
        url_hits, secret_hits, write_call_hits = _scan(body)
        if not url_hits and not secret_hits and not write_call_hits:
            out_rows.append(Row(
                workspace_id=wid, workspace_name=wname,
                notebook_id=iid, display_name=name,
                part_path=None, url=None, direction=None, source_kind=None,
                secret_type=None, match_snippet=None,
                dest_workspace_id=None, dest_workspace_name=None,
                cross_workspace=None,
                error=None, scanned_at=now,
            ))
            continue
        for path, url, direction, source_kind in url_hits:
            d_id, d_name, d_cross = _resolve_dest(url, wid, wname)
            out_rows.append(Row(
                workspace_id=wid, workspace_name=wname,
                notebook_id=iid, display_name=name,
                part_path=path, url=url,
                direction=direction, source_kind=source_kind,
                secret_type=None, match_snippet=None,
                dest_workspace_id=d_id, dest_workspace_name=d_name,
                cross_workspace=d_cross,
                error=None, scanned_at=now,
            ))
        for path, secret_type, snippet, source_kind in secret_hits:
            out_rows.append(Row(
                workspace_id=wid, workspace_name=wname,
                notebook_id=iid, display_name=name,
                part_path=path, url=None,
                direction=None, source_kind=source_kind,
                secret_type=secret_type, match_snippet=snippet,
                dest_workspace_id=None, dest_workspace_name=None,
                cross_workspace=None,
                error=None, scanned_at=now,
            ))
        for path, sig, line, source_kind in write_call_hits:
            out_rows.append(Row(
                workspace_id=wid, workspace_name=wname,
                notebook_id=iid, display_name=name,
                part_path=path, url=None,
                direction="potential_write", source_kind=source_kind,
                secret_type=sig, match_snippet=line,
                dest_workspace_id=None, dest_workspace_name=None,
                cross_workspace=None,
                error=None, scanned_at=now,
            ))
    return iter(out_rows)


# --- Lakehouse-mode partition scanner ---------------------------------------
# Each input row already has the file's bytes (loaded by spark.read.binaryFile
# in the previous cell). We just decode + scan; no REST calls, no LROs.
_LH_WS_BC  = SOURCE_LAKEHOUSE_WORKSPACE_ID or "lakehouse"
_LH_WSN_BC = (SOURCE_LAKEHOUSE_WORKSPACE_NAME
              or SOURCE_LAKEHOUSE_WORKSPACE_ID
              or "lakehouse")


def scan_partition_lakehouse(rows):
    """Runs on each Spark executor. Decodes + scans each pre-loaded file."""
    import base64  # noqa: F401  (kept for symmetry with fetch_partition)
    import json
    import os
    import re
    from datetime import datetime, timezone
    from pyspark.sql import Row

    URL_RE = re.compile(
        r"(?i)\b(?:https?|ftps?|file|abfss?|wasbs?|s3a?|s3n|gs|dbfs|adl|hdfs)://"
        r"[^\s'\"<>`)\]}]+"
        r"|\bwww\.[^\s'\"<>`)\]}]+\.[^\s'\"<>`)\]}]+"
    )
    TRAILING = ".,;:!?)]}>\"'`"

    READ_RE = re.compile(
        r"(?:"
        r"\brequests?\.(?:get|head|options)\s*\("
        r"|\burllib(?:\.request)?\.urlopen\s*\("
        r"|\burlretrieve\s*\("
        r"|\b(?:pd|pandas|pl|polars)\.(?:read|scan)_\w+\s*\("
        r"|\.read_\w+\s*\("
        r"|\.scan_(?:parquet|csv|ndjson|delta|ipc|arrow)\s*\("
        r"|\bspark\.read\b"
        r"|\b(?:httpx|aiohttp)\.(?:get|head|options)\s*\("
        r"|\bsession\.(?:get|head|options)\s*\("
        r"|\bfetch\s*\("
        r"|\bsc\.textFile\s*\("
        r"|\bspark\.sparkContext\.textFile\s*\("
        r"|!\s*wget\b"
        r"|!\s*curl\b(?![^\n]*-X\s+(?:POST|PUT|PATCH|DELETE))"
        r"|\bdownload\w*\s*\("
        r")",
        re.IGNORECASE,
    )

    WRITE_RE = re.compile(
        r"(?:"
        r"\brequests?\.(?:post|put|patch|delete)\s*\("
        r"|\b(?:httpx|aiohttp)\.(?:post|put|patch|delete)\s*\("
        r"|\bsession\.(?:post|put|patch|delete)\s*\("
        r"|\b(?:pd|pandas)\.to_\w+\s*\("
        r"|\.to_(?:csv|json|parquet|sql|excel|html|hdf|pickle|orc|feather)\s*\("
        r"|\.write_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|database)\s*\("
        r"|\.write\.\w+"
        r"|\.save(?:AsTable|AsTextFile)?\s*\("
        r"|\bupload\w*\s*\("
        r"|\bput_object\s*\("
        r"|!\s*curl[^\n]*-X\s+(?:POST|PUT|PATCH|DELETE)"
        r"|!\s*curl[^\n]*(?:--upload-file|--data\b|-d\s)"
        r")",
        re.IGNORECASE,
    )

    WRITE_CALL_RE = re.compile(
        r"(?:"
        r"\.(?:write|to)_(?:delta|parquet|csv|json|ndjson|ipc|arrow|avro|excel|feather|html|sql|orc|hdf|pickle|stata|xml|database|table|gbq)\s*\("
        r"|\.write(?:Stream)?\.(?:format|mode|option|partitionBy|bucketBy|trigger|outputMode|start|save|saveAsTable|insertInto|csv|parquet|json|orc|text|delta|jdbc)\s*\("
        r"|\b(?:requests|httpx|aiohttp|session|client|http)\.(?:post|put|patch|delete)\s*\("
        r"|\b(?:upload_blob|upload_file|upload_fileobj|upload_from_string|upload_from_file|upload_data|put_object|put_block|put_block_blob|put_item|put_record|create_blob_from_\w+)\s*\("
        r"|\bmssparkutils\.fs\.(?:cp|put|append|mv)\s*\("
        r"|\.execute\s*\(\s*[\'\"](?:INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO|TRUNCATE\s+|DROP\s+|CREATE\s+TABLE|ALTER\s+TABLE)"
        r"|\bspark\.sql\s*\(\s*f?[\'\"](?:INSERT\s+INTO|UPDATE\s+|DELETE\s+FROM|MERGE\s+INTO|TRUNCATE\s+|DROP\s+TABLE|CREATE\s+TABLE|REPLACE\s+TABLE|ALTER\s+TABLE)"
        r"|\bopen\s*\([^)]*[\'\"](?:w|wb|w\+|wb\+|a|ab|a\+|ab\+|x|xb|x\+|xb\+|r\+|rb\+)[\'\"]"
        r"|!\s*curl[^\n]*-X\s+(?:POST|PUT|PATCH|DELETE)"
        r"|!\s*curl[^\n]*(?:--upload-file|--data\b|-d\s)"
        r")",
        re.IGNORECASE,
    )

    SECRET_PATTERNS = [
        ("Storage Account Key",
         re.compile(r'AccountKey\s*=\s*[A-Za-z0-9+/=]{40,}', re.IGNORECASE)),
        ("SAS Token",
         re.compile(r'(\?|&)sig=[A-Za-z0-9%+/=]{20,}', re.IGNORECASE)),
        ("SAS Token (assigned)",
         re.compile(r'["\']https?://[^"\s]+\.blob\.core\.windows\.net[^"\s]*sig=[^"\s]+["\']',
                    re.IGNORECASE)),
        ("SQL Password",
         re.compile(r'(Password|pwd)\s*=\s*[^;\s"]{4,}', re.IGNORECASE)),
        ("Cosmos DB Key",
         re.compile(r'(AccountKey|cosmosdb[_-]?key)\s*[=:]\s*["\']?[A-Za-z0-9+/=]{40,}',
                    re.IGNORECASE)),
        ("API Key",
         re.compile(r'(api[_-]?key|apikey|subscription[_-]?key|Ocp-Apim-Subscription-Key)\s*[=:]\s*["\']?[A-Za-z0-9]{16,}',
                    re.IGNORECASE)),
        ("Client Secret",
         re.compile(r'(client[_-]?secret|app[_-]?secret)\s*[=:]\s*["\']?[A-Za-z0-9~._-]{16,}',
                    re.IGNORECASE)),
        ("Connection String",
         re.compile(r'(connection[_-]?string|conn[_-]?str)\s*[=:]\s*["\'][^"]{20,}["\']',
                    re.IGNORECASE)),
        ("Hardcoded Bearer Token",
         re.compile(r'["\']Bearer\s+eyJ[A-Za-z0-9_-]{20,}', re.IGNORECASE)),
        ("Event Hub / Service Bus Key",
         re.compile(r'SharedAccessKey\s*=\s*[A-Za-z0-9+/=]{30,}', re.IGNORECASE)),
        ("OpenAI API Key",
         re.compile(r'(openai[_-]?api[_-]?key|OPENAI_API_KEY)\s*[=:]\s*["\']?sk-[A-Za-z0-9]{20,}',
                    re.IGNORECASE)),
        ("Azure OpenAI Key",
         re.compile(r'(azure[_-]?openai[_-]?key|AZURE_OPENAI_KEY)\s*[=:]\s*["\']?[A-Fa-f0-9]{32}',
                    re.IGNORECASE)),
        ("Generic Secret Assignment",
         re.compile(r'(secret|password|passwd|token|credential)\s*[=:]\s*["\'][^"\s]{8,}["\']',
                    re.IGNORECASE)),
        ("Spark Config Account Key",
         re.compile(r'spark\.conf\.set\s*\([^)]*account\.key[^)]*\)', re.IGNORECASE)),
        ("JDBC Password",
         re.compile(r'jdbc:[a-z]+://[^\s]*password=[^;\s"]{4,}', re.IGNORECASE)),
    ]

    def _clean(u):
        while u and u[-1] in TRAILING:
            u = u[:-1]
        return u

    def _redact(text, head=4, tail=4):
        n = len(text)
        if n <= head + tail + 3:
            return "*" * n
        return text[:head] + ("*" * (n - head - tail)) + text[-tail:]

    def _extract(text):
        seen, out = set(), []
        for m in URL_RE.finditer(text):
            s = m.start()
            u = _clean(m.group(0))
            if not u or u in seen:
                continue
            seen.add(u)
            out.append((u, s, s + len(u)))
        return out

    def _find_secrets(text):
        seen, out = set(), []
        for name, rx in SECRET_PATTERNS:
            for m in rx.finditer(text):
                snip = _redact(m.group(0))
                key = (name, snip)
                if key in seen:
                    continue
                seen.add(key)
                out.append((name, snip))
        return out

    def _find_write_calls(text):
        """Destination-agnostic write-call detection. Returns deduped
        [(signature, line_snippet)] tuples.
        """
        if not text:
            return []
        seen, out = set(), []
        for m in WRITE_CALL_RE.finditer(text):
            s = m.start()
            line_start = text.rfind("\n", 0, s) + 1
            line_end = text.find("\n", m.end())
            if line_end == -1:
                line_end = len(text)
            line = text[line_start:line_end].strip()
            if len(line) > 200:
                line = line[:200] + "..."
            sig = m.group(0).strip()
            key = (sig.lower(), line)
            if key in seen:
                continue
            seen.add(key)
            out.append((sig, line))
        return out

    def _classify(text, url_start, window=200):
        ws = max(0, url_start - window)
        pre = text[ws:url_start]
        r_pos = w_pos = -1
        for m in READ_RE.finditer(pre):
            r_pos = m.start()
        for m in WRITE_RE.finditer(pre):
            w_pos = m.start()
        if r_pos >= 0 and w_pos >= 0:
            if r_pos > w_pos:
                return "read"
            if w_pos > r_pos:
                return "write"
            return "read_write"
        if r_pos >= 0:
            return "read"
        if w_pos >= 0:
            return "write"
        return "unknown"

    _WORKSPACE_URL_RE = re.compile(
        r"(?:"
        r"abfss?://([^@/\s]+)@onelake\.dfs\.fabric\.microsoft\.com"
        r"|https?://[^/\s]*\.fabric\.microsoft\.com/(?:[^?\s]*?/)?(?:v1/)?workspaces/([0-9a-f-]{8,})"
        r"|https?://(?:app|api)\.powerbi\.com/(?:[^/\s]*/)*?groups/([0-9a-f-]{8,})"
        r")",
        re.IGNORECASE,
    )
    _GUID_RE = re.compile(
        r"^[0-9a-f]{8}-?[0-9a-f]{4}-?[0-9a-f]{4}-?[0-9a-f]{4}-?[0-9a-f]{12}$",
        re.IGNORECASE,
    )
    _NAME_BY_ID = _WS_NAME_BY_ID_BC or {}
    _ID_BY_NAME = _WS_ID_BY_NAME_BC or {}

    def _resolve_dest(url, src_wid, src_wname):
        """Return (dest_id, dest_name, cross_workspace) tuple."""
        if not url:
            return None, None, None
        m = _WORKSPACE_URL_RE.search(url)
        if not m:
            return None, None, None
        raw = m.group(1) or m.group(2) or m.group(3)
        if not raw:
            return None, None, None
        if _GUID_RE.match(raw):
            dest_id = raw.lower()
            dest_name = _NAME_BY_ID.get(dest_id, "") or _NAME_BY_ID.get(raw, "")
        else:
            dest_name = raw
            dest_id = _ID_BY_NAME.get(raw.lower(), "")
        src_id_l = (src_wid or "").lower()
        src_name_l = (src_wname or "").lower()
        if dest_id and src_id_l and _GUID_RE.match(src_id_l):
            cross = (dest_id != src_id_l)
        elif dest_name and src_name_l:
            cross = (dest_name.lower() != src_name_l)
        else:
            cross = None
        return (dest_id or None), (dest_name or None), cross

    def _scan_one_file(path, raw):
        """Single-file version. Returns (url_hits, secret_hits, write_call_hits) where:
            url_hits        -> [(path, url, direction, source_kind), ...]
            secret_hits     -> [(path, secret_type, snippet, source_kind), ...]
            write_call_hits -> [(path, signature, line_snippet, source_kind), ...]
        """
        url_hits = []
        secret_hits = []
        write_call_hits = []
        lower = path.lower()
        if lower.endswith(".md"):
            for url, _s, _e in _extract(raw):
                url_hits.append((path, url, "reference", "markdown"))
            for name, snip in _find_secrets(raw):
                secret_hits.append((path, name, snip, "markdown"))
            return url_hits, secret_hits, write_call_hits
        if lower.endswith(".py"):
            for url, s, _e in _extract(raw):
                url_hits.append((path, url, _classify(raw, s), "code"))
            for name, snip in _find_secrets(raw):
                secret_hits.append((path, name, snip, "code"))
            for sig, line in _find_write_calls(raw):
                write_call_hits.append((path, sig, line, "code"))
            return url_hits, secret_hits, write_call_hits
        if lower.endswith(".ipynb"):
            try:
                nb = json.loads(raw)
            except Exception:
                for url, s, _e in _extract(raw):
                    url_hits.append((path, url, "unknown", "code"))
                for name, snip in _find_secrets(raw):
                    secret_hits.append((path, name, snip, "code"))
                for sig, line in _find_write_calls(raw):
                    write_call_hits.append((path, sig, line, "code"))
                return url_hits, secret_hits, write_call_hits
            for cell in nb.get("cells", []) or []:
                ctype = cell.get("cell_type", "")
                src = cell.get("source", "")
                src_text = ("".join(src) if isinstance(src, list)
                            else str(src))
                if ctype == "markdown":
                    for url, _s, _e in _extract(src_text):
                        url_hits.append((path, url, "reference", "markdown"))
                    for name, snip in _find_secrets(src_text):
                        secret_hits.append((path, name, snip, "markdown"))
                    continue
                if ctype == "code":
                    for url, s, _e in _extract(src_text):
                        url_hits.append((path, url,
                                         _classify(src_text, s), "code"))
                    for name, snip in _find_secrets(src_text):
                        secret_hits.append((path, name, snip, "code"))
                    for sig, line in _find_write_calls(src_text):
                        write_call_hits.append((path, sig, line, "code"))
                    for out in cell.get("outputs", []) or []:
                        if not isinstance(out, dict):
                            continue
                        out_chunks = []
                        t = out.get("text", "")
                        out_chunks.append("".join(t) if isinstance(t, list)
                                          else str(t))
                        data = out.get("data") or {}
                        if isinstance(data, dict):
                            for mime, payload in data.items():
                                if mime.startswith(("text/", "application/")):
                                    out_chunks.append(
                                        "".join(payload)
                                        if isinstance(payload, list)
                                        else str(payload)
                                    )
                        out_text = "\n".join(out_chunks)
                        for url, _s, _e in _extract(out_text):
                            url_hits.append((path, url, "reference", "output"))
                        for name, snip in _find_secrets(out_text):
                            secret_hits.append((path, name, snip, "output"))
                    continue
                for url, _s, _e in _extract(src_text):
                    url_hits.append((path, url, "unknown", "code"))
                for name, snip in _find_secrets(src_text):
                    secret_hits.append((path, name, snip, "code"))
                for sig, line in _find_write_calls(src_text):
                    write_call_hits.append((path, sig, line, "code"))
            return url_hits, secret_hits, write_call_hits
        # Unknown extension: best-effort.
        for url, s, _e in _extract(raw):
            url_hits.append((path, url, _classify(raw, s), "code"))
        for name, snip in _find_secrets(raw):
            secret_hits.append((path, name, snip, "code"))
        for sig, line in _find_write_calls(raw):
            write_call_hits.append((path, sig, line, "code"))
        return url_hits, secret_hits, write_call_hits

    now = datetime.now(timezone.utc)
    out_rows = []
    for row in rows:
        # binaryFile schema: path, modificationTime, length, content (bytes)
        path = row["path"]
        content = row["content"]
        name = os.path.basename(path) if path else ""
        try:
            raw = (bytes(content).decode("utf-8", errors="ignore")
                   if content is not None else "")
        except Exception as e:
            out_rows.append(Row(
                workspace_id=_LH_WS_BC, workspace_name=_LH_WSN_BC,
                notebook_id=path, display_name=name,
                part_path=path, url=None, direction=None, source_kind=None,
                secret_type=None, match_snippet=None,
                dest_workspace_id=None, dest_workspace_name=None,
                cross_workspace=None,
                error=f"decode: {type(e).__name__}: {e}", scanned_at=now,
            ))
            continue
        try:
            url_hits, secret_hits, write_call_hits = _scan_one_file(path, raw)
        except Exception as e:
            out_rows.append(Row(
                workspace_id=_LH_WS_BC, workspace_name=_LH_WSN_BC,
                notebook_id=path, display_name=name,
                part_path=path, url=None, direction=None, source_kind=None,
                secret_type=None, match_snippet=None,
                dest_workspace_id=None, dest_workspace_name=None,
                cross_workspace=None,
                error=f"scan: {type(e).__name__}: {e}", scanned_at=now,
            ))
            continue
        if not url_hits and not secret_hits and not write_call_hits:
            out_rows.append(Row(
                workspace_id=_LH_WS_BC, workspace_name=_LH_WSN_BC,
                notebook_id=path, display_name=name,
                part_path=path, url=None, direction=None, source_kind=None,
                secret_type=None, match_snippet=None,
                dest_workspace_id=None, dest_workspace_name=None,
                cross_workspace=None,
                error=None, scanned_at=now,
            ))
            continue
        for ppath, url, direction, source_kind in url_hits:
            d_id, d_name, d_cross = _resolve_dest(url, _LH_WS_BC, _LH_WSN_BC)
            out_rows.append(Row(
                workspace_id=_LH_WS_BC, workspace_name=_LH_WSN_BC,
                notebook_id=path,
                display_name=name, part_path=ppath, url=url,
                direction=direction, source_kind=source_kind,
                secret_type=None, match_snippet=None,
                dest_workspace_id=d_id, dest_workspace_name=d_name,
                cross_workspace=d_cross,
                error=None, scanned_at=now,
            ))
        for ppath, secret_type, snippet, source_kind in secret_hits:
            out_rows.append(Row(
                workspace_id=_LH_WS_BC, workspace_name=_LH_WSN_BC,
                notebook_id=path,
                display_name=name, part_path=ppath, url=None,
                direction=None, source_kind=source_kind,
                secret_type=secret_type, match_snippet=snippet,
                dest_workspace_id=None, dest_workspace_name=None,
                cross_workspace=None,
                error=None, scanned_at=now,
            ))
        for ppath, sig, line, source_kind in write_call_hits:
            out_rows.append(Row(
                workspace_id=_LH_WS_BC, workspace_name=_LH_WSN_BC,
                notebook_id=path,
                display_name=name, part_path=ppath, url=None,
                direction="potential_write", source_kind=source_kind,
                secret_type=sig, match_snippet=line,
                dest_workspace_id=None, dest_workspace_name=None,
                cross_workspace=None,
                error=None, scanned_at=now,
            ))
    return iter(out_rows)


result_schema = StructType([
    StructField("workspace_id", StringType(), False),
    StructField("workspace_name", StringType(), True),
    StructField("notebook_id", StringType(), False),
    StructField("display_name", StringType(), True),
    StructField("part_path", StringType(), True),
    StructField("url", StringType(), True),
    StructField("direction", StringType(), True),
    StructField("source_kind", StringType(), True),
    StructField("secret_type", StringType(), True),
    StructField("match_snippet", StringType(), True),
    StructField("dest_workspace_id", StringType(), True),
    StructField("dest_workspace_name", StringType(), True),
    StructField("cross_workspace", BooleanType(), True),
    StructField("error", StringType(), True),
    StructField("scanned_at", TimestampType(), False),
])

if SOURCE_MODE == "lakehouse":
    results_rdd = inventory_df.rdd.mapPartitions(scan_partition_lakehouse)
else:
    results_rdd = inventory_df.rdd.mapPartitions(fetch_partition)
results_df = spark.createDataFrame(results_rdd, schema=result_schema)


In [ ]:
# === Persist + summarize =====================================================
writer = (results_df.write
            .mode("overwrite")
            .option("overwriteSchema", "true"))

if WRITE_TO_DEFAULT_LAKEHOUSE:
    writer.saveAsTable(output_target)
    sql_ref = output_target
else:
    writer.format("delta").save(output_target)
    spark.read.format("delta").load(output_target).createOrReplaceTempView("_findings")
    sql_ref = "_findings"

print(f"Findings persisted to: {output_target}")

display(spark.sql(f"""
    SELECT
      COUNT(DISTINCT notebook_id)                                          AS notebooks_scanned,
      SUM(CASE WHEN url IS NOT NULL THEN 1 ELSE 0 END)                     AS total_urls,
      COUNT(DISTINCT CASE WHEN url IS NOT NULL THEN notebook_id END)       AS notebooks_with_urls,
      SUM(CASE WHEN secret_type IS NOT NULL THEN 1 ELSE 0 END)             AS total_secrets,
      COUNT(DISTINCT CASE WHEN secret_type IS NOT NULL THEN notebook_id END) AS notebooks_with_secrets,
      SUM(CASE WHEN direction = 'potential_write' THEN 1 ELSE 0 END)       AS potential_write_calls,
      COUNT(DISTINCT CASE WHEN direction = 'potential_write' THEN notebook_id END) AS notebooks_with_potential_writes,
      SUM(CASE WHEN cross_workspace = TRUE THEN 1 ELSE 0 END)              AS cross_workspace_urls,
      COUNT(DISTINCT CASE WHEN cross_workspace = TRUE THEN notebook_id END) AS notebooks_with_cross_ws,
      SUM(CASE WHEN error IS NOT NULL THEN 1 ELSE 0 END)                   AS error_rows,
      SUM(CASE WHEN direction = 'read'       THEN 1 ELSE 0 END)            AS url_reads,
      SUM(CASE WHEN direction = 'write'      THEN 1 ELSE 0 END)            AS url_writes,
      SUM(CASE WHEN direction = 'read_write' THEN 1 ELSE 0 END)            AS url_read_writes,
      SUM(CASE WHEN direction = 'reference'  THEN 1 ELSE 0 END)            AS url_references,
      SUM(CASE WHEN direction = 'unknown'    THEN 1 ELSE 0 END)            AS url_unknown
    FROM {sql_ref}
"""))

# Breakdown by direction + source_kind so you can spot, e.g., a markdown link
# vs an actual requests.get(...) call.
display(spark.sql(f"""
    SELECT direction, source_kind, COUNT(*) AS occurrences,
           COUNT(DISTINCT notebook_id) AS notebooks
    FROM {sql_ref}
    WHERE url IS NOT NULL
    GROUP BY direction, source_kind
    ORDER BY direction, source_kind
"""))

display(spark.sql(f"""
    SELECT regexp_extract(url, '^[a-z]+://([^/]+)', 1) AS host,
           direction,
           COUNT(*) AS occurrences,
           COUNT(DISTINCT notebook_id) AS notebooks
    FROM {sql_ref}
    WHERE url IS NOT NULL
    GROUP BY 1, 2
    ORDER BY occurrences DESC
    LIMIT 25
"""))

# All URLs the code is actually WRITING to (highest-signal for outbound IO).
display(spark.sql(f"""
    SELECT workspace_id, workspace_name, notebook_id, display_name, part_path, url,
           dest_workspace_id, dest_workspace_name, cross_workspace
    FROM {sql_ref}
    WHERE direction IN ('write', 'read_write')
    ORDER BY workspace_name, notebook_id
    LIMIT 200
"""))

# Cross-workspace flows: every URL whose destination workspace differs from
# the notebook's source workspace. These are the data-egress / cross-tenant
# patterns most admins want to see first.
display(spark.sql(f"""
    SELECT workspace_name AS source_workspace,
           dest_workspace_name,
           dest_workspace_id,
           direction,
           COUNT(*) AS occurrences,
           COUNT(DISTINCT notebook_id) AS notebooks
    FROM {sql_ref}
    WHERE cross_workspace = TRUE
    GROUP BY 1, 2, 3, 4
    ORDER BY occurrences DESC
    LIMIT 100
"""))

display(spark.sql(f"""
    SELECT workspace_id, workspace_name AS source_workspace, notebook_id, display_name,
           dest_workspace_id, dest_workspace_name, direction, part_path, url
    FROM {sql_ref}
    WHERE cross_workspace = TRUE
    ORDER BY workspace_name, dest_workspace_name, notebook_id
    LIMIT 200
"""))

# Secrets: counts by type, then the most exposed notebooks.
display(spark.sql(f"""
    SELECT secret_type, COUNT(*) AS occurrences,
           COUNT(DISTINCT notebook_id) AS notebooks
    FROM {sql_ref}
    WHERE secret_type IS NOT NULL
    GROUP BY secret_type
    ORDER BY occurrences DESC
"""))

display(spark.sql(f"""
    SELECT workspace_id, workspace_name, notebook_id, display_name, part_path,
           secret_type, source_kind, match_snippet
    FROM {sql_ref}
    WHERE secret_type IS NOT NULL
    ORDER BY workspace_name, notebook_id, secret_type
    LIMIT 200
"""))

# Notebook-level write-call detection: shows write API calls regardless of
# whether the destination is a literal URL. `secret_type` here holds the
# matched call signature; `match_snippet` holds the line of code.
display(spark.sql(f"""
    SELECT secret_type AS write_signature, COUNT(*) AS occurrences,
           COUNT(DISTINCT notebook_id) AS notebooks
    FROM {sql_ref}
    WHERE direction = 'potential_write'
    GROUP BY secret_type
    ORDER BY occurrences DESC
"""))

# REVIEW QUEUE: notebooks that call write APIs but yielded NO concrete write
# URL — these are the highest-value candidates for human review because the
# destination is constructed at runtime (e.g., df.write_delta(path)).
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW notebook_review_queue AS
    WITH per_nb AS (
      SELECT
        workspace_id,
        ANY_VALUE(workspace_name) AS workspace_name,
        notebook_id,
        ANY_VALUE(display_name) AS display_name,
        SUM(CASE WHEN direction = 'write'           THEN 1 ELSE 0 END) AS literal_write_urls,
        SUM(CASE WHEN direction = 'read_write'      THEN 1 ELSE 0 END) AS literal_read_write_urls,
        SUM(CASE WHEN direction = 'potential_write' THEN 1 ELSE 0 END) AS potential_writes,
        SUM(CASE WHEN cross_workspace = TRUE        THEN 1 ELSE 0 END) AS cross_workspace_urls,
        SUM(CASE WHEN secret_type IS NOT NULL
                  AND direction IS DISTINCT FROM 'potential_write' THEN 1 ELSE 0 END) AS secret_count
      FROM {sql_ref}
      GROUP BY workspace_id, notebook_id
    )
    SELECT
      workspace_id, workspace_name, notebook_id, display_name,
      potential_writes, literal_write_urls, literal_read_write_urls,
      cross_workspace_urls, secret_count,
      CASE
        WHEN potential_writes > 0 AND literal_write_urls = 0 AND literal_read_write_urls = 0
          THEN 'flag_for_review'
        WHEN cross_workspace_urls > 0
          THEN 'cross_workspace_write'
        WHEN literal_write_urls > 0 OR literal_read_write_urls > 0
          THEN 'has_literal_write_url'
        ELSE 'no_write_activity'
      END AS review_status
    FROM per_nb
    WHERE potential_writes > 0 OR literal_write_urls > 0 OR literal_read_write_urls > 0
""")

print("Review-queue view: notebook_review_queue (status = flag_for_review needs human attention).")
display(spark.sql("""
    SELECT review_status, COUNT(*) AS notebooks
    FROM notebook_review_queue
    GROUP BY review_status
    ORDER BY notebooks DESC
"""))

display(spark.sql("""
    SELECT workspace_id, workspace_name, notebook_id, display_name,
           potential_writes, cross_workspace_urls, secret_count
    FROM notebook_review_queue
    WHERE review_status = 'flag_for_review'
    ORDER BY potential_writes DESC, secret_count DESC
    LIMIT 200
"""))
